# 09b — Study Area Overview (world map + per-city insets)

One figure: a world map with the four study-area cities marked, each
connected by a callout line to an inset mini-map showing that city's
actual study-area boundary (`boundary_geojson`) and its positive
(crash) / negative (generated) point locations.

**Read-only.** Reads only `paths.yaml`'s per-city `boundary_geojson`,
`positive_points_csv`, and `negative_points_csv` -- nothing under
`src/` or `configs/` is written to. Output goes to
`OUTPUTS_DIR/paper_figures/` (the same directory notebook `09` uses),
so it lands alongside the other paper figures.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running on Colab -- skipping drive mount (paths.yaml must already resolve locally).")

In [ ]:
!pip install -q pandas numpy matplotlib seaborn pyyaml geopandas shapely cartopy

## 1. Paths and output directory

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
FIG_DIR = OUTPUTS_DIR / "paper_figures"   # same dir notebook 09 uses
FIG_DIR.mkdir(parents=True, exist_ok=True)

CITY_DISPLAY_NAME = {
    "bogor": "Bogor, Indonesia",
    "warsaw": "Warsaw, Poland",
    "krakow": "Krak\u00f3w, Poland",
    "somerville": "Somerville, USA",
}

print("Cities:", CITIES)
print("FIG_DIR:", FIG_DIR)

## 2. Load per-city boundary + positive/negative points

Each city's `boundary_geojson` gives the actual study-area polygon (used both to compute the city's map-marker position -- its centroid -- and to draw the inset outline). Points are loaded straight from `positive_points_csv`/`negative_points_csv`, no filtering beyond what those files already contain.

In [ ]:
import geopandas as gpd
import pandas as pd

city_data = {}
for city in CITIES:
    pc = paths_cfg["per_city"][city]
    entry = {}

    boundary_path = Path(pc["boundary_geojson"])
    if boundary_path.exists():
        gdf = gpd.read_file(boundary_path)
        entry["boundary"] = gdf.geometry.unary_union
        c = entry["boundary"].centroid
        entry["center"] = (c.x, c.y)  # (lon, lat)
    else:
        print(f"  [skip] {boundary_path} not found for {city}.")
        entry["boundary"] = None
        entry["center"] = None

    pos_path, neg_path = Path(pc["positive_points_csv"]), Path(pc["negative_points_csv"])
    entry["positive"] = pd.read_csv(pos_path) if pos_path.exists() else None
    entry["negative"] = pd.read_csv(neg_path) if neg_path.exists() else None
    if entry["positive"] is None:
        print(f"  [skip] {pos_path} not found for {city}.")
    if entry["negative"] is None:
        print(f"  [skip] {neg_path} not found for {city}.")

    city_data[city] = entry
    n_pos = len(entry["positive"]) if entry["positive"] is not None else 0
    n_neg = len(entry["negative"]) if entry["negative"] is not None else 0
    center = entry["center"]
    print(f"{CITY_DISPLAY_NAME.get(city, city)}: center={center}, "
          f"n_positive={n_pos}, n_negative={n_neg}")

## 3. House figure style (inline, self-contained, same as notebook 09)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


class _PaperStyle:
    FULL_W = 7.0
    FS_TICK, FS_LABEL, FS_TITLE, FS_LEGEND = 6.5, 7.5, 8.0, 6.0
    MUTED = ["#9fd4c0", "#c3b49a", "#8a7358", "#9aa4cd", "#4a4a73",
             "#8ecae0", "#f2a58c", "#3f8f7d"]
    FOCAL = "#8c2f2f"

    def apply(self, font="Liberation Sans"):
        sns.set_theme(style="ticks")
        mpl.rcParams.update({
            "font.family": "sans-serif",
            "font.sans-serif": [font, "Arial", "Helvetica", "DejaVu Sans"],
            "font.size": self.FS_TICK,
            "axes.linewidth": 0.7,
            "axes.grid": False,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "legend.frameon": False,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.02,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        })

    def save(self, fig, name):
        fig.savefig(FIG_DIR / f"{name}.pdf")
        fig.savefig(FIG_DIR / f"{name}.png")
        print(f"  [saved] {name}.pdf + {name}.png")


ps = _PaperStyle()
ps.apply()

## 4. World map + per-city inset figure

Layout: one world map (Cartopy `PlateCarree`, land/ocean/coastline
only -- no internet-tile dependency beyond Cartopy's own bundled Natural
Earth data) in the center, one inset per city at a figure corner,
connected to its map marker with a callout line
(`matplotlib.patches.ConnectionPatch`). Each inset draws the city's
actual study-area boundary polygon plus its positive (crash) and
negative (generated) points, colored consistently with `F1`'s
positive/negative palette.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import ConnectionPatch

POS_COLOR, NEG_COLOR = ps.MUTED[4], ps.FOCAL  # same pairing as F1

# Figure-fraction placement for each city's inset axes (left, bottom,
# width, height) -- one per figure corner. Assigned to match each city's
# actual map quadrant (NW/NE/SW/SE) so the callout line stays short and
# doesn't sweep across the whole map -- e.g. Bogor (SE Asia, south+east)
# goes bottom-right, not an arbitrary corner. Warsaw and Krakow are both
# northeast (Europe, ~1 degree apart), so only one can get the ideal
# top-right slot; Krakow takes bottom-left as the next-best option since
# top-left is Somerville's.
INSET_RECTS = {
    "somerville": (0.02, 0.56, 0.24, 0.24),  # NW: North America
    "warsaw": (0.74, 0.56, 0.24, 0.24),      # NE: Europe
    "bogor": (0.74, 0.06, 0.24, 0.24),       # SE: Southeast Asia
    "krakow": (0.02, 0.06, 0.24, 0.24),      # Europe, but top-right is taken
}
MAIN_RECT = (0.28, 0.12, 0.44, 0.76)


def plot_city_inset(ax, city, entry):
    boundary, pos_df, neg_df = entry["boundary"], entry["positive"], entry["negative"]
    if boundary is not None:
        bx, by = boundary.exterior.xy if boundary.geom_type == "Polygon" else ([], [])
        if bx:
            ax.plot(bx, by, color="0.3", lw=1.0, zorder=1)
        else:
            # MultiPolygon -- plot every part's exterior
            for geom in boundary.geoms:
                gx, gy = geom.exterior.xy
                ax.plot(gx, gy, color="0.3", lw=1.0, zorder=1)
    if neg_df is not None and {"lon", "lat"}.issubset(neg_df.columns):
        ax.scatter(neg_df["lon"], neg_df["lat"], s=3, color=NEG_COLOR, alpha=0.6,
                   edgecolor="none", zorder=2, label="negative")
    if pos_df is not None and {"lon", "lat"}.issubset(pos_df.columns):
        ax.scatter(pos_df["lon"], pos_df["lat"], s=3, color=POS_COLOR, alpha=0.6,
                   edgecolor="none", zorder=3, label="positive")
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor("0.5"); spine.set_linewidth(0.8)
    ax.set_aspect("equal", adjustable="datalim")
    n_pos = len(pos_df) if pos_df is not None else 0
    n_neg = len(neg_df) if neg_df is not None else 0
    ax.set_title(f"{CITY_DISPLAY_NAME.get(city, city)}\nn={n_pos} positive, {n_neg} negative",
                fontsize=ps.FS_LABEL, fontweight="bold")


fig = plt.figure(figsize=(11, 8.5))

ax_main = fig.add_axes(MAIN_RECT, projection=ccrs.PlateCarree())
ax_main.set_global()
ax_main.add_feature(cfeature.LAND, facecolor="#eee8dd", zorder=0)
ax_main.add_feature(cfeature.OCEAN, facecolor="#dbe7ef", zorder=0)
ax_main.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.4", zorder=1)
ax_main.add_feature(cfeature.BORDERS, linewidth=0.25, edgecolor="0.6", zorder=1)
ax_main.set_title("Study area cities", fontsize=ps.FS_TITLE, fontweight="bold")

for city in CITIES:
    entry = city_data[city]
    if entry["center"] is None or city not in INSET_RECTS:
        print(f"  [skip] {city} -- no boundary/center available for map placement.")
        continue
    lon, lat = entry["center"]
    ax_main.scatter([lon], [lat], s=28, color=ps.FOCAL, edgecolor="white",
                    linewidth=0.8, zorder=5, transform=ccrs.PlateCarree())

    ax_inset = fig.add_axes(INSET_RECTS[city])
    plot_city_inset(ax_inset, city, entry)

    # Callout line: main map's data point (in its own transData) -> nearest
    # corner of the inset (in the inset's own axes-fraction coords).
    rect = INSET_RECTS[city]
    inset_cx = 0.5  # connect from the inset's own edge closest to center
    inset_edge_x = 0.0 if rect[0] < MAIN_RECT[0] else 1.0
    inset_edge_y = 0.5
    con = ConnectionPatch(
        xyA=(lon, lat), coordsA=ax_main.transData,
        xyB=(inset_edge_x, inset_edge_y), coordsB=ax_inset.transAxes,
        color="0.35", lw=0.8, linestyle="--", zorder=4)
    fig.add_artist(con)

legend_handles = [
    plt.Line2D([], [], marker="o", color="none", markerfacecolor=POS_COLOR, markersize=6, label="positive (crash)"),
    plt.Line2D([], [], marker="o", color="none", markerfacecolor=NEG_COLOR, markersize=6, label="negative (generated)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, 0.0),
          fontsize=ps.FS_LEGEND, frameon=False)

ps.save(fig, "F0_study_area_overview")
plt.show()

In [ ]:
print("Figures in", FIG_DIR, ":")
for p in sorted(FIG_DIR.glob("F0_study_area_overview*")):
    print(" -", p.name)